## nb30 — Citation & Publication Lift by CORE Tier (RQ3)

Are the lift effects from Section 4.3 consistent across venue tiers, or concentrated at A* conferences?

**Data sources:**
- `data/matched/author_lift.csv` — per-author lift scores (from nb11)
- `data/matched/all_authors_within_paper.csv` — has `conference` column per author
- `data/matched/conference_core_ranks.csv` — CORE 2026 rankings

**Merge key:** `author_id` → `conference` → `core_rank`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from pathlib import Path

ROOT    = Path('..')
MATCH   = ROOT / 'data' / 'matched'
FIG_DIR = ROOT / 'data' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
lift      = pd.read_csv(MATCH / 'author_lift.csv')
authors   = pd.read_csv(MATCH / 'all_authors_within_paper.csv')
core      = pd.read_csv(MATCH / 'conference_core_ranks.csv')

print('lift cols:   ', lift.columns.tolist())
print('authors cols:', authors.columns.tolist())
print('core cols:   ', core.columns.tolist())

In [ ]:
# Keep one row per author (deduplicate — same author may appear on multiple papers)
author_conf = (
    authors[['author_id', 'conference']]
    .drop_duplicates(subset='author_id', keep='first')
)

# Normalise conference names for merge
author_conf['conference'] = author_conf['conference'].str.strip().str.upper()
core.columns = core.columns.str.strip()

# Identify the right column names in core
print(core.head(10))

In [ ]:
# Rename core columns to standard names — adjust if column names differ
# Expected: one column for conference acronym/name, one for rank
conf_col = core.columns[0]
rank_col = core.columns[1]
core = core.rename(columns={conf_col: 'conference', rank_col: 'core_rank'})
core['conference'] = core['conference'].str.strip().str.upper()

# Merge: author -> conference -> core rank
author_conf = author_conf.merge(core, on='conference', how='left')
author_conf['core_rank'] = author_conf['core_rank'].fillna('Unranked')

print(author_conf['core_rank'].value_counts())

In [ ]:
# Merge lift with tier
df = lift.merge(author_conf[['author_id', 'core_rank']], on='author_id', how='left')

# Keep only junior award authors (career_group == 'junior' or is_award == 1)
# Print available columns to pick the right filter
print(df.columns.tolist())
print(df.shape)
print(df.head(3))

In [ ]:
# Filter to junior award authors only
# Adjust column name if needed based on output above
if 'career_group' in df.columns:
    junior = df[df['career_group'] == 'junior'].copy()
elif 'group' in df.columns:
    junior = df[df['group'] == 'junior'].copy()
else:
    # fallback: use all rows
    junior = df.copy()

# Cap extreme outliers (lift > 20 are noise)
junior = junior[junior['citation_lift'] <= 20].copy()

# Standardise tier labels
tier_order = ['A*', 'A', 'Unranked']
junior['core_rank'] = junior['core_rank'].apply(
    lambda x: x if x in tier_order else 'Unranked'
)

print(junior.groupby('core_rank')[['citation_lift', 'publication_lift']].describe())

In [ ]:
# Kruskal-Wallis test across tiers
groups_cit = [junior[junior['core_rank'] == t]['citation_lift'].dropna() for t in tier_order]
groups_pub = [junior[junior['core_rank'] == t]['publication_lift'].dropna() for t in tier_order]

kw_cit = stats.kruskal(*groups_cit)
kw_pub = stats.kruskal(*groups_pub)

print(f'Citation lift  — Kruskal-Wallis H={kw_cit.statistic:.3f}, p={kw_cit.pvalue:.4f}')
print(f'Publication lift — Kruskal-Wallis H={kw_pub.statistic:.3f}, p={kw_pub.pvalue:.4f}')

# Medians per tier
print()
print(junior.groupby('core_rank')[['citation_lift', 'publication_lift']].median())

In [ ]:
COLORS = {'A*': '#00696e', 'A': '#8b3a0f', 'Unranked': '#4a4a8a'}

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax, metric, label in zip(
    axes,
    ['citation_lift', 'publication_lift'],
    ['Citation Lift', 'Publication Lift']
):
    positions = [1, 2, 3]
    for pos, tier in zip(positions, tier_order):
        data = junior[junior['core_rank'] == tier][metric].dropna()
        bp = ax.boxplot(
            data, positions=[pos], widths=0.5, patch_artist=True,
            boxprops=dict(facecolor=COLORS[tier], alpha=0.4),
            medianprops=dict(color='black', linewidth=2.5),
            whiskerprops=dict(color=COLORS[tier], linewidth=1.5),
            capprops=dict(color=COLORS[tier], linewidth=1.5),
            flierprops=dict(marker='', linestyle='none')
        )
        jitter = np.random.uniform(-0.15, 0.15, size=len(data))
        ax.scatter(pos + jitter, data, alpha=0.2, s=8, color=COLORS[tier], zorder=2)
        med = data.median()
        n   = len(data)
        ax.annotate(f'{med:.2f}\n(n={n})', xy=(pos, med),
                    xytext=(pos + 0.3, med), fontsize=8,
                    color=COLORS[tier], va='center')

    ax.axhline(1.0, color='gray', linestyle='dashed', linewidth=1, label='No change (lift=1)')
    ax.set_xticks(positions)
    ax.set_xticklabels(tier_order, fontsize=11)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(f'{label} by CORE Tier\n(Junior Award Authors)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '30_lift_by_core_tier.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 30_lift_by_core_tier.png')